## CAT_DOG_CLASSIFIER

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision

In [6]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
val_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset=datasets.ImageFolder('dataset/train',transform=train_transform)
testset=datasets.ImageFolder('dataset/test',transform=val_transform)

In [7]:
train_loader=DataLoader(trainset,batch_size=32,shuffle=True)
test_loader=DataLoader(testset,batch_size=32,shuffle=False)

In [8]:
class CatDogCNN(nn.Module):
    def __init__(self):
        super(CatDogCNN, self).__init__()

        self.convo_layers = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 75x75

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 37x37

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 18x18
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 18 * 18, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)           # 1 output for binary classification
        )

    def forward(self, x):
        x = self.convo_layers(x)
        x = self.classifier(x)
        return x

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CatDogCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
print(device)

cuda


In [18]:
epochs = 10
model.train()

best_loss = float('inf')   # initialize with very high value

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    avg_loss = epoch_training_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    # ✅ Save only if current model is better
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(),'best_cat_dog_model.pth')
        print("✅ Best model saved!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 1/10, Loss: 0.3080
✅ Best model saved!
Epoch 2/10, Loss: 0.2993
✅ Best model saved!
Epoch 3/10, Loss: 0.2891
✅ Best model saved!
Epoch 4/10, Loss: 0.2757
✅ Best model saved!
Epoch 5/10, Loss: 0.2595
✅ Best model saved!
Epoch 6/10, Loss: 0.2565
✅ Best model saved!
Epoch 7/10, Loss: 0.2402
✅ Best model saved!
Epoch 8/10, Loss: 0.2391
✅ Best model saved!
Epoch 9/10, Loss: 0.2286
✅ Best model saved!
Epoch 10/10, Loss: 0.2222
✅ Best model saved!


In [ ]:
model.load_state_dict(torch.load('best_cat_dog_model.pth'))

In [19]:
model.to(device)
model.eval()

correct_labels = 0
total_labels = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)

        probs = torch.sigmoid(outputs)
        predicted = (probs >= 0.5).float()

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

accuracy = correct_labels / total_labels * 100
print(f"Accuracy = {accuracy:.2f}%")

Accuracy = 90.98%
